# Projeto de Python para Finanças

In [ ]:
# instalação
# pip install yfinance pandas numpy plotly nbformat

### Parte 1: Obter cotações e construção de carteira

In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta 

acoes = ["ITUB4.SA", "PETR4.SA","VALE3.SA","IVVB11.SA"]
# IBOVESPA = "^BVSP"
# S&P500 = "^GSPC"
# Dólar = "BRL=X"
# Ouro = "GC=F"

indices = ["^BVSP","^GSPC","BRL=X","GC=F"]

data_incio = datetime.now() - timedelta(days=730)
data_incio = data_incio.strftime("%Y-%m-%d")
data_fim = datetime.now().strftime("%Y-%m-%d")
print(data_fim, data_incio)

def pegar_cotacoes(listas_tickers, data_incio, data_fim):
    df = yf.download(listas_tickers,data_incio,data_fim, auto_adjust=False)
    df = df["Adj Close"]
    df = df.ffill()
    df = df.dropna()
    return df
lista_cotacoes = acoes + indices
df_cotacoes = pegar_cotacoes(lista_cotacoes, data_incio, data_fim)
display(df_cotacoes)    




In [ ]:
# CARTEIRA EM TERMOS DE VALOR FINANCEIRO
dic_carteira = {
    "ITUB4.SA": 5000,
    "PETR4.SA": 4000,
    "VALE3.SA": 3000,
    "IVVB11.SA": 6000
}

df_carteira = pd.DataFrame()
total_investido = sum(dic_carteira.values())
print(total_investido)

for ativo in dic_carteira:
    preco_inicial_ativo = df_cotacoes[ativo].iloc[0]
    qtde_acoes = dic_carteira[ativo] / preco_inicial_ativo
    df_carteira[ativo] = df_cotacoes[ativo] * qtde_acoes
df_carteira["Total"] = df_carteira.sum(axis=1)
display(df_carteira)



### Parte 2: Rentabilidade e Comparação com Benchmarks

In [ ]:
df_cotacoes["SP500 (R$)"] = df_cotacoes["^GSPC"] * df_cotacoes["BRL=X"]
df_cotacoes["ouro (R$)"] = df_cotacoes["GC=F"] * df_cotacoes["BRL=X"]
df_cotacoes["Dólar"] = df_cotacoes["BRL=X"]

df_cotacoes = df_cotacoes.drop(columns=["^GSPC","GC=F","BRL=X"])
display(df_cotacoes)

In [ ]:
#valor_final = 18102
#valor_inicial = 18000
#variacao_reais = valor_final - valor_inicial
#variacao_percentual = (variacao_reais / valor_inicial) * 100
#print(f"Variação em reais: {variacao_reais:.2f}")
#print(f"Variação percentual: {variacao_percentual:.2f}%")

def calcular_retorno(df):
    retorno = df.iloc[-1] / df.iloc[0] -1
    return retorno
display(calcular_retorno(df_carteira))
display(calcular_retorno(df_cotacoes))

In [ ]:
df_comparacao = df_cotacoes.drop(columns=acoes)
df_comparacao["carteira"] = df_carteira["Total"]
display(df_comparacao)
print(calcular_retorno(df_comparacao))



In [ ]:
df_comparacao = (df_comparacao / df_comparacao.iloc[0] -1)*100
display(df_comparacao)

import plotly.express as px
grafico = px.line(df_comparacao, x=df_comparacao.index, y=df_comparacao.columns)
grafico.update_layout(template="plotly_dark")
grafico.show()







### Parte 3: Análise de Risco

In [ ]:
import numpy as np
# correlação
df_cotacoes["carteira"] = df_carteira["Total"]

#rentabilidade diária
tabela_rentabilidade_diaria = df_cotacoes / df_cotacoes.shift(1)
tabela_rentabilidade_diaria = np.log(tabela_rentabilidade_diaria).dropna()
tabela_correlacao = tabela_rentabilidade_diaria.corr()


grafico_correlacao = px.imshow(tabela_correlacao, text_auto=True, color_continuous_scale="greens")
grafico_correlacao.update_layout(template="plotly_dark")
grafico_correlacao.show()

# variância
display(tabela_rentabilidade_diaria)
tabela_volatilidade = tabela_rentabilidade_diaria.std() * np.sqrt(252)
display(tabela_volatilidade)

### Parte 4: Análise Técnica e Indicadores

In [ ]:
ticker = "VALE3.SA"
import yfinance as yf
import plotly.graph_objects as go

df = yf.download(ticker,"2020-01-01","2025-12-31",multi_level_index=False)  
display(df)
#média movel de 50 dias
df["MM50"] = df["Close"].rolling(50).mean()

#Média móvel de 200 dias
df["MM200"] = df["Close"].rolling(200).mean()


grafico = go.Figure()
grafico.add_trace(go.Candlestick(x=df.index,open=df["Open"], high=df["High"], low=df["Low"], close=df["Close"]))
#adicionando médias móveis
grafico.add_trace(go.Scatter(x=df.index, y=df["MM50"], mode="   lines", name="MM50", line=dict(color="blue", width=1)))
grafico.add_trace(go.Scatter(x=df.index, y=df["MM200"], mode="lines", name="MM200", line=dict(color="yellow", width=1)))


grafico.update_layout(template="plotly_dark")
grafico.show()